# Bollywood Movie Investment & Box Office Intelligence Platform
## Hindi Movie Box Office Prediction System

**Dataset:** 400 Hindi movies (2023–2025)

This notebook trains the final model and saves a FastAPI-compatible `model.pkl` bundle to `../app/pickle files/`.
Cell 13 verifies that the API-style pipeline reproduces the notebook predictions exactly.

## 1. Imports

In [1]:
import re
import os
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('Libraries loaded. scikit-learn =', sklearn.__version__)

Libraries loaded. scikit-learn = 1.7.2


## 2. Load Dataset (from CSV)

In [2]:
# CSV must sit in the same folder as this notebook:
# Movie_Box_Office_Project/notebook/cleaned_movies_dataset(refined).csv
CSV_PATH = 'cleaned_movies_dataset(refined).csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'CSV not found: {os.path.abspath(CSV_PATH)}\n'
        'Put the file in the same folder as the notebook, or edit CSV_PATH.'
    )

# Try common encodings (Excel-saved CSVs are often not plain utf-8)
df = None
for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin-1'):
    try:
        df = pd.read_csv(CSV_PATH, encoding=enc)
        print('Loaded with encoding:', enc)
        break
    except UnicodeDecodeError:
        continue
if df is None:
    raise ValueError('Could not read the CSV with any common encoding.')

df.columns = df.columns.str.strip()      # remove accidental spaces in headers
print('Dataset shape:', df.shape)
display(df.head())

EXPECTED_COLUMNS = [
    'Year', 'Genre', 'Director', 'Box Office Collection', 'Budget',
    'Male_Lead', 'Female_Lead', 'Production_House', 'Release type'
]
missing_columns = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}\nFound: {list(df.columns)}')
print('All required columns are present.')

Loaded with encoding: utf-8
Dataset shape: (400, 13)


,Movie_Name,Year,IMDB Rating,Genre,Director,Box Office Collection,Budget,Male_Lead,Female_Lead,Production_House,Source_URLs,Search_Status,Release type
0,Jawan,2023,6.9,Action epic,Atlee,1159.00,300,ShahRukh Khan,Nayanthara,Red Chillies Entertainment,https://en.wikipedia.org/wiki/Jawan_(film) | h...,Complete,Theatrical
1,Pathaan,2023,5.8,Action Thriller,Siddharth Anand,1052.50,250,ShahRukh Khan,Deepika Padukone,Yash Raj Films,https://en.wikipedia.org/wiki/Pathaan_(film) |...,Complete,Theatrical
2,Animal,2023,6.2,Action Drama,Andy Serkis,917.82,100,Ranbir Kapoor,Rashmika Mandanna,Aniventure The Imaginarium Studios,https://en.wikipedia.org/wiki/Animal_(2023_Ind...,Complete,Theatrical
3,Gadar 2,2023,5.1,Period Action Drama,Anil Sharma,691.08,60,Sunny Deol,Ameesha Patel,Zee Studios,https://en.wikipedia.org/wiki/Gadar_2 | https:...,Complete,Theatrical
4,Dunki,2023,6.5,Comedy-Drama,Rajkumar Hirani,470.60,120,Taapsee Pannu,Taapsee Pannu,Jio Studios Red Chillies Entertainment Rajkuma...,https://en.wikipedia.org/wiki/Dunki_(film) | h...,Complete,Theatrical


All required columns are present.


## 3. Shared Helper Functions (identical logic is used in the API)

In [3]:
TARGET = 'Box Office Collection'
RAW_INPUT_COLUMNS = [
    'Year', 'Budget', 'Director', 'Female_Lead', 'Male_Lead',
    'Production_House', 'Genre', 'Release type'
]
CAT_COLS = ['Genre', 'Director', 'Male_Lead', 'Female_Lead', 'Production_House', 'Release type']
NUM_COLS = ['Budget', 'Year']

GENRE_KEYWORDS = [
    'Action', 'Drama', 'Comedy', 'Thriller', 'Romantic', 'Romance',
    'Crime', 'Horror', 'Musical', 'Biographical', 'Mystery', 'Family',
    'Historical', 'Social', 'Spy', 'Period', 'Mythological', 'Fantasy',
    'War', 'Sports'
]

def group_genre(g):
    if pd.isna(g):
        return 'Unknown'
    tokens = re.findall(r'[A-Za-z]+', str(g))
    for t in tokens:
        for k in GENRE_KEYWORDS:
            if t.lower() == k.lower():
                return 'Romance' if k == 'Romantic' else k
    return 'Other'

def clean_raw(frame, cat_cols, num_cols):
    """Trim spaces, blank -> NaN, numeric coercion."""
    frame = frame.copy()
    for c in cat_cols:
        s = frame[c].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
        s = s.mask(s == '')
        frame[c] = s.astype('object').where(s.notna(), np.nan)
    for c in num_cols:
        frame[c] = pd.to_numeric(frame[c], errors='coerce')
    return frame

def norm_key(series):
    """Lookup key used for all frequency maps (case/space insensitive)."""
    return (series.astype(str).str.strip()
            .str.replace(r'\s+', ' ', regex=True).str.lower())

def prepare_features(raw_df, b):
    """EXACT same pipeline as the API. raw_df has RAW_INPUT_COLUMNS."""
    cat_cols = list(b['cat_imputer'].feature_names_in_)
    num_cols = list(b['num_imputer'].feature_names_in_)

    row = clean_raw(raw_df[b['raw_input_columns']], cat_cols, num_cols)
    row[cat_cols] = b['cat_imputer'].transform(row[cat_cols])
    row[num_cols] = b['num_imputer'].transform(row[num_cols])

    row['Budget'] = row['Budget'].clip(upper=b['budget_cap'])
    row['Genre_Grouped'] = row['Genre'].apply(group_genre)
    row['Budget_Log'] = np.log1p(row['Budget'].clip(lower=0))

    row['Director_Freq'] = norm_key(row['Director']).map(b['director_freq_map']).fillna(1)
    row['ProdHouse_Freq'] = norm_key(row['Production_House']).map(b['production_house_freq_map']).fillna(1)
    row['MaleLead_Freq'] = norm_key(row['Male_Lead']).map(b['male_lead_freq_map']).fillna(1)
    row['FemaleLead_Freq'] = norm_key(row['Female_Lead']).map(b['female_lead_freq_map']).fillna(1)
    row['Genre_Hist_Avg'] = row['Genre_Grouped'].map(b['genre_map']).fillna(b['global_mean'])

    return row[b['final_model_features']].copy()

print('Helpers defined.')

Helpers defined.


## 4. Target and Model Inputs

In [4]:
d = df.copy()
d[TARGET] = pd.to_numeric(d[TARGET], errors='coerce')
d = d.dropna(subset=[TARGET]).copy()
d = clean_raw(d, CAT_COLS, NUM_COLS)

X = d[RAW_INPUT_COLUMNS].copy()
y = d[TARGET].copy()
print(f'Usable rows: {len(y)}')

Usable rows: 282


## 5. Train/Test Split

In [5]:
budget_for_bins = X['Budget'].fillna(X['Budget'].median())
budget_bins = pd.qcut(budget_for_bins, q=4, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=budget_bins
)
X_train, X_test = X_train.copy(), X_test.copy()
y_train, y_test = y_train.copy(), y_test.copy()

X_test_raw = X_test.copy()      # untouched copy for the final API-equivalence check
print(f'Train={len(X_train)}, Test={len(X_test)}')

Train=225, Test=57


## 6. Missing Values and Budget Cap

In [6]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
num_imputer = SimpleImputer(strategy='median')

X_train[CAT_COLS] = cat_imputer.fit_transform(X_train[CAT_COLS])
X_test[CAT_COLS] = cat_imputer.transform(X_test[CAT_COLS])

X_train[NUM_COLS] = num_imputer.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS] = num_imputer.transform(X_test[NUM_COLS])

budget_cap = float(X_train['Budget'].quantile(0.99))
X_train['Budget'] = X_train['Budget'].clip(upper=budget_cap)
X_test['Budget'] = X_test['Budget'].clip(upper=budget_cap)
print(f'Budget cap: {budget_cap:.2f} Cr')

Budget cap: 315.20 Cr


## 7. Feature Engineering

In [7]:
X_train['Genre_Grouped'] = X_train['Genre'].apply(group_genre)
X_test['Genre_Grouped'] = X_test['Genre'].apply(group_genre)

X_train['Budget_Log'] = np.log1p(X_train['Budget'].clip(lower=0))
X_test['Budget_Log'] = np.log1p(X_test['Budget'].clip(lower=0))

# Frequency maps use normalised (lowercase) keys
director_freq_map = norm_key(X_train['Director']).value_counts().to_dict()
production_house_freq_map = norm_key(X_train['Production_House']).value_counts().to_dict()
male_lead_freq_map = norm_key(X_train['Male_Lead']).value_counts().to_dict()
female_lead_freq_map = norm_key(X_train['Female_Lead']).value_counts().to_dict()

for frame in (X_train, X_test):
    frame['Director_Freq'] = norm_key(frame['Director']).map(director_freq_map).fillna(1)
    frame['ProdHouse_Freq'] = norm_key(frame['Production_House']).map(production_house_freq_map).fillna(1)
    frame['MaleLead_Freq'] = norm_key(frame['Male_Lead']).map(male_lead_freq_map).fillna(1)
    frame['FemaleLead_Freq'] = norm_key(frame['Female_Lead']).map(female_lead_freq_map).fillna(1)

global_mean = float(y_train.mean())

def smoothed_target_encoding(col, target, k=10):
    stats = target.groupby(col).agg(['mean', 'count'])
    return ((stats['count'] * stats['mean']) + (k * global_mean)) / (stats['count'] + k)

genre_map = smoothed_target_encoding(X_train['Genre_Grouped'], y_train, k=10).to_dict()
X_train['Genre_Hist_Avg'] = X_train['Genre_Grouped'].map(genre_map).fillna(global_mean)
X_test['Genre_Hist_Avg'] = X_test['Genre_Grouped'].map(genre_map).fillna(global_mean)

print('Feature engineering completed.')

Feature engineering completed.


## 8. Final Model Features and Preprocessing

In [8]:
drop_cols = ['Genre', 'Director', 'Production_House', 'Male_Lead', 'Female_Lead', 'Budget']
X_train_sel = X_train.drop(columns=drop_cols).copy()
X_test_sel = X_test.drop(columns=drop_cols).copy()

NUM_FEATURES = [
    'Year', 'Budget_Log', 'Director_Freq', 'ProdHouse_Freq',
    'Genre_Hist_Avg', 'MaleLead_Freq', 'FemaleLead_Freq'
]
CAT_FEATURES = ['Release type', 'Genre_Grouped']

FINAL_MODEL_FEATURES = X_train_sel.columns.tolist()
print('FINAL MODEL FEATURE ORDER:', FINAL_MODEL_FEATURES)

preprocess = ColumnTransformer([
    ('num_scaled', StandardScaler(), NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
])

y_train_log = np.log1p(y_train.clip(lower=0))
y_test_log = np.log1p(y_test.clip(lower=0))

FINAL MODEL FEATURE ORDER: ['Year', 'Release type', 'Genre_Grouped', 'Budget_Log', 'Director_Freq', 'ProdHouse_Freq', 'MaleLead_Freq', 'FemaleLead_Freq', 'Genre_Hist_Avg']


## 9. Model Comparison

In [9]:
def bias_variance_label(train_r2, test_r2):
    if train_r2 < 0.3 and test_r2 < 0.3:
        return 'Underfit'
    if (train_r2 - test_r2) > 0.15:
        return 'Overfit'
    return 'Goodfit'

models = {
    'LR': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=5000),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVM': SVR(kernel='rbf', C=10),
    'DT': DecisionTreeRegressor(max_depth=5, random_state=42),
    'RF': RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42),
    'GB': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
}

results = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
X_full_sel = pd.concat([X_train_sel, X_test_sel], ignore_index=True)
y_full_log = pd.concat([y_train_log, y_test_log], ignore_index=True)

for name, estimator in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', estimator)])
    pipe.fit(X_train_sel, y_train_log)
    pred_train = pipe.predict(X_train_sel)
    pred_test = pipe.predict(X_test_sel)

    train_r2 = r2_score(y_train_log, pred_train)
    test_r2 = r2_score(y_test_log, pred_test)
    train_rmse = mean_squared_error(y_train, np.maximum(0, np.expm1(pred_train))) ** 0.5
    test_rmse = mean_squared_error(y_test, np.maximum(0, np.expm1(pred_test))) ** 0.5
    test_mae = mean_absolute_error(y_test, np.maximum(0, np.expm1(pred_test)))
    cv_scores = cross_val_score(pipe, X_full_sel, y_full_log, cv=cv, scoring='r2')

    results.append([name, round(train_rmse, 2), round(test_rmse, 2), round(test_mae, 2),
                    round(train_r2, 3), round(test_r2, 3),
                    bias_variance_label(train_r2, test_r2), round(cv_scores.mean(), 3)])

results_df = pd.DataFrame(results, columns=['Model', 'TrainRMSE', 'TestRMSE', 'TestMAE',
                                            'TrainR2', 'TestR2', 'Fit', 'CV_R2_mean'])
display(results_df.sort_values('CV_R2_mean', ascending=False))

,Model,TrainRMSE,TestRMSE,TestMAE,TrainR2,TestR2,Fit,CV_R2_mean
2,Lasso,134.30,174.00,50.12,0.739,0.726,Goodfit,0.680
6,RF,92.20,191.34,57.62,0.902,0.674,Overfit,0.659
1,Ridge,136.37,172.99,52.34,0.754,0.685,Goodfit,0.645
4,SVM,90.63,198.42,58.27,0.878,0.645,Overfit,0.629
0,LR,134.56,197.97,61.78,0.757,0.637,Goodfit,0.609
7,GB,54.14,278.00,90.43,0.942,0.544,Overfit,0.564
5,DT,97.81,248.00,81.74,0.844,0.622,Overfit,0.534
3,KNN,134.10,194.11,53.77,0.715,0.577,Goodfit,0.527


## 10. Hyperparameter Tuning — Random Forest

In [10]:
rf_pipe = Pipeline([('prep', preprocess), ('model', RandomForestRegressor(random_state=42))])

rf_param_grid = {
    'model__n_estimators': [200, 300, 500],
    'model__max_depth': [4, 6, 8, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(rf_pipe, rf_param_grid, cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train_sel, y_train_log)

best_rf_model = rf_grid.best_estimator_
print('Best params:', rf_grid.best_params_)
print('Best CV R²:', rf_grid.best_score_)

Best params: {'model__max_depth': 4, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 500}
Best CV R²: 0.7182776090225947


## 11. Final Model Evaluation

In [11]:
rf_pred_log = best_rf_model.predict(X_test_sel)
rf_pred = np.maximum(0, np.expm1(rf_pred_log))

print(f'Test R² (log target): {r2_score(y_test_log, rf_pred_log):.3f}')
print(f'Test RMSE (Cr): {mean_squared_error(y_test, rf_pred) ** 0.5:.2f}')
print(f'Test MAE  (Cr): {mean_absolute_error(y_test, rf_pred):.2f}')

Test R² (log target): 0.698
Test RMSE (Cr): 184.83
Test MAE  (Cr): 50.69


## 12. Save FastAPI-Compatible Model Bundle

In [12]:
MODEL_DIR = '../app/pickle files'      # notebook/ -> app/pickle files
os.makedirs(MODEL_DIR, exist_ok=True)

model_bundle = {
    'model': best_rf_model,
    'model_name': 'Random Forest',
    'target_column': TARGET,
    'raw_input_columns': RAW_INPUT_COLUMNS,
    'cat_imputer': cat_imputer,
    'num_imputer': num_imputer,
    'budget_cap': budget_cap,
    'director_freq_map': director_freq_map,
    'production_house_freq_map': production_house_freq_map,
    'male_lead_freq_map': male_lead_freq_map,
    'female_lead_freq_map': female_lead_freq_map,
    'genre_map': genre_map,
    'global_mean': global_mean,
    'final_model_features': FINAL_MODEL_FEATURES,
    'sklearn_version': sklearn.__version__,
}

model_path = os.path.join(MODEL_DIR, 'model.pkl')
joblib.dump(model_bundle, model_path)
print('MODEL SAVED:', os.path.abspath(model_path))

MODEL SAVED: /Users/deepakm/Documents/ML internship/Movie_Box_Office_Project/app/pickle files/model.pkl


## 13. Verify the API Pipeline Gives the Same Predictions

In [13]:
loaded = joblib.load(model_path)

required_keys = [
    'model', 'model_name', 'target_column', 'raw_input_columns',
    'cat_imputer', 'num_imputer', 'budget_cap',
    'director_freq_map', 'production_house_freq_map',
    'male_lead_freq_map', 'female_lead_freq_map',
    'genre_map', 'global_mean', 'final_model_features'
]
missing = [k for k in required_keys if k not in loaded]
if missing:
    raise ValueError(f'Missing bundle keys: {missing}')

# Raw test rows -> same code path as the API
X_api = prepare_features(X_test_raw, loaded)
api_pred = np.maximum(0, np.expm1(loaded['model'].predict(X_api)))

print('Max difference vs notebook predictions:', np.abs(api_pred - rf_pred).max())
assert np.allclose(api_pred, rf_pred, atol=1e-6), 'API pipeline does NOT match notebook!'
print('✅ API-style predictions match the notebook exactly.')

# Sample prediction
sample = pd.DataFrame([{
    'Year': 2025, 'Budget': 150, 'Director': 'Rohit Shetty',
    'Female_Lead': 'Deepika Padukone', 'Male_Lead': 'Ajay Devgn',
    'Production_House': 'Reliance Entertainment', 'Genre': 'Action',
    'Release type': X_test_raw['Release type'].iloc[0]
}])
print('Sample prediction (Cr):',
      round(float(np.expm1(loaded['model'].predict(prepare_features(sample, loaded)))[0]), 2))

Max difference vs notebook predictions: 0.0
✅ API-style predictions match the notebook exactly.
Sample prediction (Cr): 125.7
